In [ ]:
# ---------------------------------------------------------------------
# FULL SCRIPT: PASTE ALL OF THIS CODE INTO A SINGLE COLAB CELL AND RUN IT
# ---------------------------------------------------------------------

#@title A-to-Z Training Script for Neural Codec (TS3 GAN)
#@markdown ### 1. Install Dependencies
#@markdown This cell will install all required packages and define all models and training functions.
!pip install pesq[speechmetrics] pystoi librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import os
import numpy as np
import torch.nn.functional as F
import math
import traceback
import tarfile
import requests
import threading
import librosa # Added for metric calculation in validation
from pesq import pesq # Added for metric calculation in validation
from pystoi import stoi # Added for metric calculation in validation
from torch.optim.lr_scheduler import ExponentialLR
from google.colab import files, drive # Ensure drive is imported

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---------------------------------------------------------------------
# GOOGLE DRIVE MOUNT
# ---------------------------------------------------------------------
print("\n--- Mounting Google Drive for Checkpointing ---")
drive.mount('/content/drive')
print("Google Drive mounted successfully.")
print("---------------------------------------------")

# ---------------------------------------------------------------------
# CONFIGURATION OPTIONS
# ---------------------------------------------------------------------
#@markdown ---
#@markdown ### 2. Training Configuration
#@markdown Training uses the Adversarial Codec (GACodec) for high quality.
MODEL_ARCHITECTURE = "TS3_Codec (16kbps, Transformer GAN)" #@param ["TS3_Codec (16kbps, Transformer GAN)"]
DATASET_TO_DOWNLOAD = "train-clean-100" #@param ["dev-clean", "train-clean-100", "train-clean-360"]
#@markdown *Note: Use `train-clean-100` for best results. Training GANs requires larger data.*

#@markdown ---
#@markdown ### 3. Training Hyperparameters (CRITICAL TUNING FOR PESQ > 3.6)
EPOCHS = 1000 #@param {type:"integer"}
#@markdown *Set to 1000 for realistic high-quality training. Will be stopped by session limit.*
LEARNING_RATE = 0.0001 #@param {type:"number"}

#@markdown **Adversarial and Feature Loss Weights**
#@markdown *These are key for quality.*
LAMBDA_STFT = 45.0 #@param {type:"number"}
LAMBDA_FM = 4.0 #@param {type:"number"}
#@markdown *(Increased from 2.0 to 4.0 to force better perceptual feature matching)*

#@markdown **GAN Stability Controls**
DISC_WARMUP_STEPS = 100 #@param {type:"integer"}
LR_DECAY_RATE = 0.9999 #@param {type:"number"}
#@markdown **Discriminator to Generator Training Ratio**
D_G_RATIO = 2 #@param {type:"integer"}
#@markdown *(Train Discriminator 2x for every 1x Generator to maintain balance)*

#@markdown ---
#@markdown ### 4. Hardware & VRAM Settings
BATCH_SIZE = 8 #@param {type:"integer"}
#@markdown *Note: Increased model complexity may require even smaller batch sizes (4 or 6).*
GRAD_ACCUM_STEPS = 4 #@param {type:"integer"}
USE_AMP = True #@param {type:"boolean"}
NUM_WORKERS = 4 #@param {type:"integer"}

#@markdown **Latency Control (Crucial for <20ms end-to-end)**
TFM_HISTORY_CHUNKS = 0 #@param {type:"integer"}
#@markdown *SET TO 0: Guarantees 10ms frame latency (160 samples). Set to 1 (20ms total) or 2 (30ms total) if quality plateau is hit.*

#@markdown ---
#@markdown ### 5. File Paths & Checkpointing
SAVE_PATH_PREFIX = "low_latency_codec" #@param {type:"string"}
DOWNLOAD_PATH = "/content/LibriSpeech" #@param {type:"string"}
GOOGLE_DRIVE_CHECKPOINT_PATH = "/content/drive/MyDrive/Codec_Checkpoints" #@param {type:"string"}
#@markdown *Crucial: This path is used for real-time saving and loading checkpoints.*
#@markdown ---

# ---------------------------------------------------------------------
# DATASET DOWNLOAD FUNCTION
# ---------------------------------------------------------------------

def download_librispeech(dataset_name, path):
    """Downloads and extracts a LibriSpeech dataset."""
    url_map = {
        "dev-clean": "https://www.openslr.org/resources/12/dev-clean.tar.gz",
        "train-clean-100": "https://www.openslr.org/resources/12/train-clean-100.tar.gz",
        "train-clean-360": "https://www.openslr.org/resources/12/train-clean-360.tar.gz",
    }

    datasets_to_check = [dataset_name]
    if dataset_name != "dev-clean":
        datasets_to_check.append("dev-clean")

    downloaded_paths = {}

    for name in datasets_to_check:
        if name not in url_map: continue

        url = url_map[name]
        filename = url.split("/")[-1]
        compressed_path = os.path.join(path, filename)
        target_folder_name = name
        expected_dataset_path = os.path.join(path, "LibriSpeech", target_folder_name)

        print(f"Checking for existing dataset folder for {name} at: {expected_dataset_path}")
        if os.path.exists(expected_dataset_path) and os.path.isdir(expected_dataset_path):
            print(f"Dataset {name} already found.")
            downloaded_paths[name] = expected_dataset_path
            continue

        try:
            if not os.path.exists(compressed_path):
                print(f"Downloading {filename}...")
                with requests.get(url, stream=True) as r:
                    r.raise_for_status()

                    os.makedirs(os.path.dirname(compressed_path), exist_ok=True)

                    with open(compressed_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            f.write(chunk)
                print("Download complete.")

            print(f"Extracting {name}...")
            with tarfile.open(compressed_path, "r:gz") as tar:
                tar.extractall(path=path)
            print("Extraction complete.")
            os.remove(compressed_path)

            if os.path.exists(expected_dataset_path) and os.path.isdir(expected_dataset_path):
                print(f"✓ Dataset {name} ready.")
                downloaded_paths[name] = expected_dataset_path
            else:
                print(f"ERROR: Could not find extracted folder for {name}.")

        except Exception as e:
            print(f"Error during download/extraction for {name}: {e}")
            traceback.print_exc()

    if dataset_name not in downloaded_paths:
        return None, None

    return downloaded_paths.get(dataset_name), downloaded_paths.get("dev-clean")


# ---------------------------------------------------------------------
# MODEL DEFINITIONS
# ---------------------------------------------------------------------

# --- Global Configuration (Unchanged) ---
HOP_SIZE = 160 # 10ms frame (160 samples / 16000 Hz = 0.01s) - CRITICAL LATENCY FIX
LATENT_DIM = 128 # High capacity

VQ_EMBEDDINGS = 256
NUM_VQ_STAGES = 2
VQ_INDICES_PER_STAGE = 10
NUM_QUANTIZERS = VQ_INDICES_PER_STAGE

# --- Loss Components (Unchanged) ---
class MultiResolutionSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[1024, 2048, 512], hop_sizes=[120, 240, 50], win_lengths=[600, 1200, 240]):
        super(MultiResolutionSTFTLoss, self).__init__()
        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        self.win_lengths = win_lengths
        self.window = torch.hann_window
        assert len(fft_sizes) == len(hop_sizes) == len(win_lengths)

    def forward(self, y_hat, y):
        sc_loss, mag_loss = 0.0, 0.0
        for fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            window = self.window(win, device=y.device)
            y_hat_float = y_hat.squeeze(1).to(torch.float32)
            y_float = y.squeeze(1).to(torch.float32)
            spec_hat = torch.stft(y_hat_float, n_fft=fft, hop_length=hop, win_length=win, window=window, return_complex=True)
            spec = torch.stft(y_float, n_fft=fft, hop_length=hop, win_length=win, window=window, return_complex=True)

            sc_loss += torch.norm(torch.abs(spec) - torch.abs(spec_hat), p='fro') / (torch.norm(torch.abs(spec), p='fro') + 1e-6)
            mag_loss += F.l1_loss(torch.log(torch.abs(spec).clamp(min=1e-9)), torch.log(torch.abs(spec_hat).clamp(min=1e-9)))

        return (sc_loss / len(self.fft_sizes)) + (mag_loss / len(self.fft_sizes))

# --- Causal Components (Unchanged) ---
class CausalConv1d(nn.Conv1d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.causal_padding = self.kernel_size[0] - 1
    def forward(self, x):
        return super().forward(F.pad(x, (self.causal_padding, 0)))

class CausalConvTranspose1d(nn.ConvTranspose1d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.causal_padding = self.kernel_size[0] - self.stride[0]
    def forward(self, x):
        x = super().forward(x)
        if self.causal_padding != 0:
            return x[..., :-self.causal_padding]
        return x

# --- Residual Vector Quantizer (RVQ) (Unchanged) ---
class ResidualVectorQuantizer(nn.Module):
    def __init__(self, num_stages, num_embeddings, embedding_dim, commitment_cost):
        super().__init__()
        self.num_stages = num_stages
        self.commitment_cost = commitment_cost
        self.vqs = nn.ModuleList([
            VectorQuantizerSingle(num_embeddings, embedding_dim, 0.0)
            for _ in range(num_stages)
        ])

    def forward(self, z_e):
        quantized_output = 0.0
        residual = z_e
        total_vq_loss = 0.0
        all_indices = []

        for vq in self.vqs:
            z_q_i, vq_loss_i, indices_i = vq(residual)
            residual = residual - z_q_i.detach()
            quantized_output = quantized_output + z_q_i
            total_vq_loss += vq_loss_i
            all_indices.append(indices_i)

        total_vq_loss = total_vq_loss * self.commitment_cost / self.num_stages
        return quantized_output, total_vq_loss, torch.stack(all_indices, dim=1)

class VectorQuantizerSingle(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / self.num_embeddings, 1.0 / self.num_embeddings)

    def forward(self, z_e):
        z_e_float = z_e.to(torch.float32)
        z_e_flat = z_e_float.permute(0, 2, 1).contiguous().view(-1, self.embedding_dim)
        distances = (torch.sum(z_e_flat**2, dim=1, keepdim=True)
                     + torch.sum(self.embedding.weight**2, dim=1)
                     - 2 * torch.matmul(z_e_flat, self.embedding.weight.t()))
        encoding_indices = torch.argmin(distances, dim=1)
        z_q = self.embedding(encoding_indices).view(z_e_float.shape[0], -1, self.embedding_dim)
        z_q = z_q.permute(0, 2, 1).contiguous()
        e_loss = F.mse_loss(z_q.detach(), z_e_float)
        z_q = z_e + (z_q - z_e_float).detach()
        return z_q, e_loss, encoding_indices.view(z_e.shape[0], -1)


# --- Encoder/Decoder (Unchanged - Architecturally Sound for 16kbps/10ms) ---
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        ResBlock = lambda c: nn.Sequential(CausalConv1d(c, c, 3), nn.ELU(), CausalConv1d(c, c, 1))
        # Total Downsampling: 16x. (160 samples -> 10 latent vectors)
        self.net = nn.Sequential(
            CausalConv1d(1, 64, 5), nn.ELU(), ResBlock(64),
            CausalConv1d(64, 128, 3, stride=2), nn.ELU(), ResBlock(128), # x2
            CausalConv1d(128, 256, 3, stride=2), nn.ELU(), ResBlock(256), # x4
            CausalConv1d(256, 512, 3, stride=2), nn.ELU(), ResBlock(512), # x8
            CausalConv1d(512, 512, 3, stride=2), nn.ELU(), ResBlock(512), # x16 total stride
            CausalConv1d(512, LATENT_DIM, 3, stride=1), nn.ELU(),
        )
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        ResBlock = lambda c: nn.Sequential(CausalConv1d(c, c, 3), nn.ELU(), CausalConv1d(c, c, 1))
        # Total Upsampling: x16. (10 latent vectors -> 160 samples)
        self.net = nn.Sequential(
            CausalConvTranspose1d(LATENT_DIM, 512, 3, stride=1), nn.ELU(), ResBlock(512),
            CausalConvTranspose1d(512, 512, 3, stride=2), nn.ELU(), ResBlock(512), # 10 -> 20
            CausalConvTranspose1d(512, 256, 3, stride=2), nn.ELU(), ResBlock(256), # 20 -> 40
            CausalConvTranspose1d(256, 128, 3, stride=2), nn.ELU(), ResBlock(128), # 40 -> 80
            CausalConvTranspose1d(128, 64, 3, stride=2), nn.ELU(), ResBlock(64), # 80 -> 160 (x16 total)
            CausalConv1d(64, 1, 3), nn.Tanh()
        )
    def forward(self, x): return self.net(x)

# --- Causal Transformer (CRITICAL CPU LATENCY FIX) ---
class CausalTransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, num_layers=1, history_chunks=0):
        super().__init__()
        self.d_model = d_model
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, activation=F.gelu)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.state_len = history_chunks * VQ_INDICES_PER_STAGE
        print(f"CausalTransformerEncoder initialized with STATE_LEN: {self.state_len}, Layers: {num_layers}")


    def get_causal_mask(self, sz, device):
        return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).to(torch.bool)

    def forward(self, x, state):
        if self.state_len > 0 and state is not None:
            state_history = state[:, -self.state_len:, :]
            inp = torch.cat([state_history, x], dim=1)
        else:
            inp = x

        new_state = inp.detach()
        mask = self.get_causal_mask(inp.shape[1], x.device)
        norm_inp = self.norm(inp)
        out = self.transformer(norm_inp, mask=mask, src_key_padding_mask=None)
        out = out[:, -x.shape[1]:, :]
        return out, new_state


# --- Generator: TS3 Codec (OPTIMIZED RVQ Version) ---
class TS3_Codec(nn.Module):
    """The Generator model now utilizing RVQ for increased capacity."""
    def __init__(self, history_chunks=0):
        super().__init__()
        self.encoder = Encoder()
        self.quantizer = ResidualVectorQuantizer(NUM_VQ_STAGES, VQ_EMBEDDINGS, LATENT_DIM, 0.25)
        self.encoder_tfm = CausalTransformerEncoder(LATENT_DIM, nhead=4, num_layers=1, history_chunks=history_chunks)
        self.decoder_tfm = CausalTransformerEncoder(LATENT_DIM, nhead=4, num_layers=1, history_chunks=history_chunks)
        self.decoder = Decoder()

    def forward(self, x, h_enc=None, h_dec=None):
        x = x.to(torch.float32)
        z_e = self.encoder(x)
        z_e_tfm_in = z_e.permute(0, 2, 1)
        z_e_tfm_out, h_enc_new = self.encoder_tfm(z_e_tfm_in, h_enc)
        z_e_tfm_out = z_e_tfm_out.permute(0, 2, 1)
        z_q, vq_loss, indices = self.quantizer(z_e_tfm_out)
        z_q_tfm_in = z_q.permute(0, 2, 1)
        z_q_tfm_out, h_dec_new = self.decoder_tfm(z_q_tfm_in, h_dec)
        z_q_tfm_out = z_q_tfm_out.permute(0, 2, 1)
        x_hat = self.decoder(z_q_tfm_out)
        return x_hat, vq_loss, (h_enc_new, h_dec_new)

    def encode(self, x, h_enc):
        """Streaming encode path."""
        x = x.to(torch.float32)
        z_e = self.encoder(x)
        z_e_tfm_in = z_e.permute(0, 2, 1)
        z_e_tfm_out, h_enc_new = self.encoder_tfm(z_e_tfm_in, h_enc)
        z_e_tfm_out = z_e_tfm_out.permute(0, 2, 1)
        with torch.no_grad():
            _, _, indices = self.quantizer(z_e_tfm_out)
        return indices.view(indices.shape[0], -1), h_enc_new

    def decode(self, indices, h_dec):
        """Streaming decode path - reconstructs VQ vectors from flattened indices."""
        indices = indices.view(indices.shape[0], NUM_VQ_STAGES, VQ_INDICES_PER_STAGE)
        z_q = 0.0
        for stage in range(NUM_VQ_STAGES):
            vq_layer = self.quantizer.vqs[stage]
            indices_i = indices[:, stage, :]
            z_q_i = vq_layer.embedding(indices_i)
            z_q = z_q + z_q_i.permute(0, 2, 1)

        z_q_tfm_in = z_q.permute(0, 2, 1)
        z_q_tfm_out, h_dec_new = self.decoder_tfm(z_q_tfm_in, h_dec)
        z_q_tfm_out = z_q_tfm_out.permute(0, 2, 1)
        x_hat = self.decoder(z_q_tfm_out)
        return x_hat, h_dec_new


# --- Discriminator: Multi-Scale Adversarial Network (Unchanged) ---
class Discriminator(nn.Module):
    """Single Discriminator operating on one scale/resolution."""
    def __init__(self, start_channels=16):
        super().__init__()
        self.net = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(1, start_channels, 15, stride=1, padding=7),
                nn.LeakyReLU(0.2),
            ),
            nn.Sequential(nn.Conv1d(start_channels, start_channels * 2, 41, stride=4, padding=20, groups=4), nn.LeakyReLU(0.2)),
            nn.Sequential(nn.Conv1d(start_channels * 2, start_channels * 4, 41, stride=4, padding=20, groups=16), nn.LeakyReLU(0.2)),
            nn.Sequential(nn.Conv1d(start_channels * 4, start_channels * 8, 41, stride=4, padding=20, groups=64), nn.LeakyReLU(0.2)),
            nn.Sequential(nn.Conv1d(start_channels * 8, start_channels * 8, 5, stride=1, padding=2), nn.LeakyReLU(0.2)),
        ])
        self.final_conv = nn.Conv1d(start_channels * 8, 1, 3, stride=1, padding=1)

    def forward(self, x):
        feature_maps = []
        for layer in self.net:
            x = layer(x)
            feature_maps.append(x)
        output = self.final_conv(x)
        feature_maps.append(output)
        return output, feature_maps

class MultiScaleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.discriminators = nn.ModuleList([
            Discriminator(start_channels=16),
            Discriminator(start_channels=32),
            Discriminator(start_channels=64),
        ])
        self.downsample = nn.AvgPool1d(kernel_size=4, stride=2, padding=1, count_include_pad=False)

    def forward(self, x):
        outputs = []
        feature_maps = []

        out_d1, fm_d1 = self.discriminators[0](x)
        outputs.append(out_d1); feature_maps.append(fm_d1)

        x_2x = self.downsample(x)
        out_d2, fm_d2 = self.discriminators[1](x_2x)
        outputs.append(out_d2); feature_maps.append(fm_d2)

        x_4x = self.downsample(x_2x)
        out_d3, fm_d3 = self.discriminators[2](x_4x)
        outputs.append(out_d3); feature_maps.append(fm_d3)

        return outputs, feature_maps


# --- TRADITIONAL CODECS (Unchanged) ---
class MuLawCodec:
    def __init__(self, quantization_channels=256): self.mu = float(quantization_channels - 1)
    def encode(self, x):
        mu_t = torch.tensor(self.mu, device=x.device, dtype=torch.float32)
        encoded = torch.sign(x) * torch.log1p(mu_t * torch.abs(x)) / torch.log1p(mu_t)
        return (((encoded + 1) / 2 * self.mu) + 0.5).to(torch.uint8)
    def decode(self, z):
        z_float = z.to(torch.float32)
        mu_t = torch.tensor(self.mu, device=z.device, dtype=torch.float32)
        y = (z_float / self.mu) * 2.0 - 1.0
        return (torch.sign(y) * (1.0 / self.mu) * (torch.pow(1.0 + self.mu, torch.abs(y)) - 1.0)).unsqueeze(1)

class ALawCodec:
    def __init__(self): self.A = 87.6
    def encode(self, x):
        a_t = torch.tensor(self.A, device=x.device, dtype=torch.float32)
        abs_x = torch.abs(x)
        encoded = torch.zeros_like(x)
        cond = abs_x < (1 / self.A)
        encoded[cond] = torch.sign(x[cond]) * (a_t * abs_x[cond]) / (1 + torch.log(a_t))
        encoded[~cond] = torch.sign(x[~cond]) * (1 + torch.log(a_t * abs_x[~cond])) / (1 + torch.log(a_t))
        return (((encoded + 1) / 2 * 255) + 0.5).to(torch.uint8)
    def decode(self, z):
        z_float = z.to(torch.float32)
        a_t = torch.tensor(self.A, device=z.device, dtype=torch.float32)
        y = (z_float / 127.5) - 1.0
        abs_y = torch.abs(y)
        decoded = torch.zeros_like(y)
        cond = abs_y < (1 / (1 + torch.log(a_t)))
        decoded[cond] = torch.sign(y[cond]) * (abs_y[cond] * (1 + torch.log(a_t))) / a_t
        decoded[~cond] = torch.sign(y[~cond]) * torch.exp(abs_y[~cond] * (1 + torch.log(a_t)) - 1) / a_t
        return decoded.unsqueeze(1)

# --- DATASET & TRAINING LOGIC ---
TRAIN_CHUNK_SIZE = 16000

class AudioChunkDataset(Dataset):
    def __init__(self, directory, chunk_size=TRAIN_CHUNK_SIZE, sample_rate=16000):
        self.chunk_size, self.sample_rate = chunk_size, sample_rate
        self.file_paths = [os.path.join(r, f) for r, _, fs in os.walk(directory) for f in fs if f.lower().endswith(('.wav', '.flac'))]
        if not self.file_paths: raise ValueError("No audio files found.")
    def __len__(self): return len(self.file_paths)
    def __getitem__(self, idx):
        try:
            waveform, sr = torchaudio.load(self.file_paths[idx])
            if sr != self.sample_rate: waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform)
            if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            waveform = waveform.to(torch.float32)
            if waveform.shape[1] > self.chunk_size:
                start = np.random.randint(0, waveform.shape[1] - self.chunk_size)
                waveform = waveform[:, start:start + self.chunk_size]
            else:
                waveform = F.pad(waveform, (0, self.chunk_size - waveform.shape[1]))
            if waveform.shape[1] != self.chunk_size:
                waveform = F.pad(waveform, (0, self.chunk_size - waveform.shape[1]))
            return waveform
        except Exception as e:
            print(f"Warning: Skipping file {self.file_paths[idx]}. Error: {e}")
            return torch.zeros((1, self.chunk_size), dtype=torch.float32)


def generator_loss(disc_fake_features, disc_real_features_for_fm, fm_weight, gan_weight):
    """Calculates Generator Loss (Adversarial + Feature Matching)."""
    adv_loss = 0.0
    fm_loss = 0.0

    # Adversarial Loss (LSGAN/MSE Loss for Generator: Target real=1.0)
    for disc_fake in disc_fake_features:
        adv_loss += F.mse_loss(disc_fake[-1], torch.ones_like(disc_fake[-1]))

    # Feature Matching Loss (L1 distance between intermediate discriminator features)
    for (fake_features, real_features) in zip(disc_fake_features, disc_real_features_for_fm):
        for (fake_feature, real_feature) in zip(fake_features[:-1], real_features[:-1]):
            fm_loss += F.l1_loss(fake_feature, real_feature.detach())

    # Normalize losses
    num_disc = len(disc_fake_features)
    num_fm_layers = sum(len(f) - 1 for f in disc_fake_features)
    fm_loss = fm_loss / (num_fm_layers if num_fm_layers > 0 else 1.0)
    adv_loss = adv_loss / num_disc

    return adv_loss * gan_weight, fm_loss * fm_weight

def discriminator_loss(disc_real_features, disc_fake_features):
    """Calculates Discriminator Loss (Real vs Fake)."""
    loss = 0.0
    # LSGAN/MSE Loss for Discriminator: Target real=1.0, target fake=0.0
    for real_out, fake_out in zip(disc_real_features, disc_fake_features):
        real_loss = F.mse_loss(real_out[-1], torch.ones_like(real_out[-1]))
        fake_loss = F.mse_loss(fake_out[-1], torch.zeros_like(fake_out[-1]))
        loss += (real_loss + fake_loss)
    return loss / len(disc_real_features)

# --- Metric Calculation Utility ---
def calculate_metrics(model, dataloader, device, progress_callback, sr=16000):
    """Calculates PESQ and STOI on a subset of the DEV data."""
    model.eval()
    total_pesq, total_stoi, total_samples = 0, 0, 0
    max_test_batches = 10

    test_data_iter = iter(dataloader)

    try:
        for batch_idx in range(max_test_batches):
            inputs = next(test_data_iter).to(device)

            with torch.no_grad():
                h_enc, h_dec = None, None
                x_hat, _, _ = model(inputs, h_enc, h_dec)

                x_hat = x_hat[..., :inputs.shape[-1]]

                original_np = inputs.squeeze().cpu().numpy()
                reconstructed_np = x_hat.squeeze().cpu().numpy()

                for i in range(original_np.shape[0]):
                    orig = original_np[i]
                    rec = reconstructed_np[i]

                    try:
                        pesq_score = pesq(sr, orig, rec, 'wb')
                        stoi_score = stoi(orig, rec, sr, extended=False)

                        total_pesq += pesq_score
                        total_stoi += stoi_score
                        total_samples += 1
                    except Exception as e:
                        progress_callback.emit(f"Warning: Metric calculation failed for one sample: {e}")
                        continue

        if total_samples > 0:
            avg_pesq = total_pesq / total_samples
            avg_stoi = total_stoi / total_samples
            progress_callback.emit(f"--- VALIDATION (DEV SET) --- PESQ: {avg_pesq:.3f} | STOI: {avg_stoi:.3f} ---")

        model.train()
    except StopIteration:
        progress_callback.emit("Warning: Reached end of validation dataloader during metric calculation.")
        model.train()


def train_model(train_dataset_path, val_dataset_path, epochs, learning_rate, batch_size, save_path_prefix, google_drive_path, progress_callback, stop_event, model_type, use_amp=True, accum_steps=1, num_workers=4, disc_warmup_steps=100, lr_decay_rate=0.9999, tfm_history_chunks=0, lambda_stft=45.0, lambda_fm=4.0, d_g_ratio=2):
    """
    Main GAN Training function with checkpoint loading/saving and validation.
    """
    if model_type != 'transformer':
        progress_callback.emit("Internal Error: Training only supports 'transformer' model_type now."); return

    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        progress_callback.emit(f"Using device: {device}")

        # Training Data Setup
        train_dataset = AudioChunkDataset(directory=train_dataset_path)
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
        progress_callback.emit(f"Train Dataset loaded with {len(train_dataset)} files.")

        # Validation Data Setup
        val_dataset = AudioChunkDataset(directory=val_dataset_path)
        val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=1, pin_memory=True, drop_last=True)
        progress_callback.emit(f"Validation Dataset (dev-clean) loaded with {len(val_dataset)} files.")


        # ----------------------------------------
        # MODEL, OPTIMIZER, SCALER INITIALIZATION
        # ----------------------------------------
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        generator = TS3_Codec(history_chunks=tfm_history_chunks).to(device)
        discriminator = MultiScaleDiscriminator().to(device)

        optimizer_g = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.9))
        optimizer_d = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.9))

        scheduler_g = ExponentialLR(optimizer_g, gamma=lr_decay_rate)
        scheduler_d = ExponentialLR(optimizer_d, gamma=lr_decay_rate)

        stft_criterion = MultiResolutionSTFTLoss().to(device)
        l1_criterion = nn.L1Loss().to(device)

        progress_callback.emit(f"Starting GAN training for TS3_Codec (Generator) and Discriminator...")

        lambda_l1, lambda_vq = 1.0, 0.1
        gan_weight, fm_weight = 1.0, lambda_fm

        # Default starting values
        start_epoch = 0
        global_step = 0
        d_step_counter = 0

        # ----------------------------------------
        # GOOGLE DRIVE & CHECKPOINTING SETUP
        # ----------------------------------------
        # Create the checkpoint directory if it doesn't exist
        os.makedirs(google_drive_path, exist_ok=True)

        checkpoint_file = os.path.join(google_drive_path, f"{save_path_prefix}_full_checkpoint.pt")
        final_pth_file = os.path.join(google_drive_path, f"{save_path_prefix}_generator_final.pth")

        # --- Check for Checkpoint to Resume ---
        if os.path.exists(checkpoint_file):
            progress_callback.emit(f"Found checkpoint: {checkpoint_file}. Resuming training...")
            checkpoint = torch.load(checkpoint_file, map_location=device)

            # Load states
            generator.load_state_dict(checkpoint['generator_state_dict'])
            discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
            optimizer_g.load_state_dict(checkpoint['optimizer_g_state_dict'])
            optimizer_d.load_state_dict(checkpoint['optimizer_d_state_dict'])
            scheduler_g.load_state_dict(checkpoint['scheduler_g_state_dict'])
            scheduler_d.load_state_dict(checkpoint['scheduler_d_state_dict'])

            if use_amp and checkpoint['scaler_state_dict'] is not None:
                scaler.load_state_dict(checkpoint['scaler_state_dict'])

            start_epoch = checkpoint['epoch'] + 1
            global_step = checkpoint['global_step']
            d_step_counter = checkpoint.get('d_step_counter', 0)

            progress_callback.emit(f"Resumed from Epoch {start_epoch}, Global Step {global_step}.")
        else:
            progress_callback.emit(f"No checkpoint found at {checkpoint_file}. Starting from scratch.")


        # Initial validation check (Epoch 0)
        if start_epoch == 0:
            calculate_metrics(generator, val_dataloader, device, progress_callback)

        for epoch in range(start_epoch, epochs):
            if stop_event.is_set():
                progress_callback.emit("Training stopped by user."); break

            generator.train()
            discriminator.train()

            for i, data in enumerate(train_dataloader):
                if stop_event.is_set(): break
                inputs = data.to(device)
                global_step += 1
                d_step_counter += 1

                # --- 2. TRAIN DISCRIMINATOR (MSD) ---
                if global_step > disc_warmup_steps and d_step_counter % d_g_ratio == 0:
                    optimizer_d.zero_grad()

                    with torch.cuda.amp.autocast(enabled=use_amp):
                        h_enc, h_dec = None, None
                        x_hat, _, _ = generator(inputs, h_enc, h_dec)
                        input_len = inputs.shape[-1]
                        if x_hat.shape[-1] > input_len:
                            x_hat = x_hat[..., :input_len]
                        elif x_hat.shape[-1] < input_len:
                            x_hat = F.pad(x_hat, (0, input_len - x_hat.shape[-1]))

                        x_hat_detached = x_hat.detach()

                        _, disc_real_features = discriminator(inputs)
                        _, disc_fake_features = discriminator(x_hat_detached)
                        loss_d_total = discriminator_loss(disc_real_features, disc_fake_features)

                    scaler.scale(loss_d_total / accum_steps).backward()

                    if global_step % accum_steps == 0:
                        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
                        scaler.step(optimizer_d)
                        scaler.update()
                        optimizer_d.zero_grad()
                        scheduler_d.step()

                # --- 1. TRAIN GENERATOR (TS3_Codec) ---
                if global_step > disc_warmup_steps:
                    optimizer_g.zero_grad()

                    with torch.cuda.amp.autocast(enabled=use_amp):
                        x_hat, vq_loss, _ = generator(inputs, h_enc, h_dec)

                        input_len = inputs.shape[-1]
                        if x_hat.shape[-1] > input_len:
                            x_hat = x_hat[..., :input_len]
                        elif x_hat.shape[-1] < input_len:
                            x_hat = F.pad(x_hat, (0, input_len - x_hat.shape[-1]))

                        stft_loss = stft_criterion(x_hat, inputs)
                        l1_loss = l1_criterion(x_hat, inputs)

                        disc_fake_outputs_g, disc_fake_features_g = discriminator(x_hat)
                        with torch.no_grad():
                            _, disc_real_features_for_fm = discriminator(inputs)

                        adv_loss, fm_loss = generator_loss(disc_fake_features_g, disc_real_features_for_fm, fm_weight=fm_weight, gan_weight=gan_weight)

                        loss_g_total = ((adv_loss + fm_loss) +
                                            lambda_stft * stft_loss +
                                            lambda_l1 * l1_loss +
                                            lambda_vq * vq_loss)

                    scaler.scale(loss_g_total / accum_steps).backward()

                    if global_step % accum_steps == 0:
                        torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
                        scaler.step(optimizer_g)
                        scaler.update()
                        optimizer_g.zero_grad()
                        scheduler_g.step()

                    if global_step % (20 * accum_steps * d_g_ratio) == 19:
                        progress_callback.emit(
                            f"[E {epoch + 1}, B {i + 1}] G Loss: {loss_g_total.item():.4f} (Adv: {adv_loss.item():.4f}, FM: {fm_loss.item():.4f}, STFT: {stft_loss.item():.4f}) | D Loss: {loss_d_total.item():.4f} | LR: {optimizer_g.param_groups[0]['lr']:.2e}"
                        )

            # --- CRITICAL VALIDATION STEP ---
            progress_callback.emit(f"--- Epoch {epoch + 1} finished ---")
            if (epoch + 1) % 10 == 0:
                calculate_metrics(generator, val_dataloader, device, progress_callback)

            # --- CRITICAL CHECKPOINT SAVE (Real-time GDrive Save) ---
            progress_callback.emit(f"Saving epoch {epoch + 1} checkpoints to Google Drive...")

            # 1. Save Full Checkpoint (for resume)
            full_state = {
                'epoch': epoch,
                'global_step': global_step,
                'd_step_counter': d_step_counter,
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'optimizer_g_state_dict': optimizer_g.state_dict(),
                'optimizer_d_state_dict': optimizer_d.state_dict(),
                'scheduler_g_state_dict': scheduler_g.state_dict(),
                'scheduler_d_state_dict': scheduler_d.state_dict(),
                'scaler_state_dict': scaler.state_dict() if use_amp else None,
            }
            torch.save(full_state, checkpoint_file)
            progress_callback.emit(f"Full Checkpoint saved to: {checkpoint_file}")

            # 2. Save Generator .pth file (for deployment/transfer)
            torch.save(generator.state_dict(), final_pth_file)
            progress_callback.emit(f"Generator weights saved to: {final_pth_file}")

            progress_callback.emit(f"Checkpoints for Epoch {epoch + 1} saved successfully.")


        if not stop_event.is_set():
            progress_callback.emit("Training finished. Saving final Generator (TS3_Codec) weights...")
            torch.save(generator.state_dict(), final_pth_file)
            progress_callback.emit(f"Final Generator weights saved to: {final_pth_file}")
            progress_callback.emit(f"The full checkpoint is also available at: {checkpoint_file}")

    except Exception as e:
        progress_callback.emit(f"ERROR in training: {e}")
        progress_callback.emit(traceback.format_exc())

    finally:
        if 'generator' in locals() and generator is not None:
             try:
                 progress_callback.emit(f"Attempting final save of Generator weights to {final_pth_file}...")
                 torch.save(generator.state_dict(), final_pth_file)
                 progress_callback.emit(f"Final Generator weights state saved successfully.")
             except Exception as final_save_e:
                 progress_callback.emit(f"!!! Error during final save: {final_save_e} !!!")


# ---------------------------------------------------------------------
# MAIN EXECUTION SCRIPT
# ---------------------------------------------------------------------

print("Starting main training script...")

# 1. Map GUI string to internal model_type
model_type_internal = "transformer"

# 2. Update save path
print(f"Selected model: {MODEL_ARCHITECTURE}")
print(f"Checkpoints will be saved to: {GOOGLE_DRIVE_CHECKPOINT_PATH}")

# 3. Download data (fetches main data and dev-clean)
print(f"Downloading/Verifying data in {DOWNLOAD_PATH}...")
train_dataset_path, val_dataset_path = download_librispeech(DATASET_TO_DOWNLOAD, DOWNLOAD_PATH)

if train_dataset_path and val_dataset_path:
    print(f"Train Dataset ready at: {train_dataset_path}")
    print(f"Validation Dataset ready at: {val_dataset_path}")

    # 4. Start training
    print("--- Starting Training ---")
    stop_event = threading.Event()

    class ColabProgressEmitter:
        def emit(self, msg):
            print(msg)

    colab_printer = ColabProgressEmitter()

    train_model(
        train_dataset_path=train_dataset_path,
        val_dataset_path=val_dataset_path, # Pass the separate validation path
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        save_path_prefix=SAVE_PATH_PREFIX, # Pass prefix to train_model
        google_drive_path=GOOGLE_DRIVE_CHECKPOINT_PATH, # Pass GDrive path to train_model
        progress_callback=colab_printer,
        stop_event=stop_event,
        model_type=model_type_internal,
        use_amp=USE_AMP,
        accum_steps=GRAD_ACCUM_STEPS,
        num_workers=NUM_WORKERS,
        # Pass new stability params
        disc_warmup_steps=DISC_WARMUP_STEPS,
        lr_decay_rate=LR_DECAY_RATE,
        tfm_history_chunks=TFM_HISTORY_CHUNKS, # Pass latency control
        # Pass new quality params
        lambda_stft=LAMBDA_STFT,
        lambda_fm=LAMBDA_FM,
        d_g_ratio=D_G_RATIO
    )

    print("--- Training Finished ---")
    print("\n=======================================================")
    print("Training is complete. Files saved in Google Drive:")
    print(f"Full Checkpoint for Resume: {os.path.join(GOOGLE_DRIVE_CHECKPOINT_PATH, f'{SAVE_PATH_PREFIX}_full_checkpoint.pt')}")
    print(f"Final Model Weights (.pth): {os.path.join(GOOGLE_DRIVE_CHECKPOINT_PATH, f'{SAVE_PATH_PREFIX}_generator_final.pth')}")
    print("=======================================================")

else:
    print("Failed to download/find dataset. Training aborted.")


PyTorch Version: 2.8.0+cu126
CUDA Available: True
GPU: Tesla T4
Starting main training script...
Selected model: GRU_Codec (16kbps, Fast)
Model will be saved to: low_latency_codec_gru.pth
Downloading/Verifying train-clean-100...
Checking for existing dataset folder at: /content/LibriSpeech/LibriSpeech/train-clean-100
Dataset already found at: /content/LibriSpeech/LibriSpeech/train-clean-100
Dataset ready at: /content/LibriSpeech/LibriSpeech/train-clean-100
--- Starting Training ---
Using device: cuda
Searching for audio files in: /content/LibriSpeech/LibriSpeech/train-clean-100
Found 28539 audio files.
Dataset loaded with 28539 files. Using chunk size: 16000
Starting training for GRU_Codec with FIXED VQ...
Using StepLR scheduler: step_size=10, gamma=0.7


/tmp/ipython-input-3905292067.py:529: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_s

[Epoch 1, Batch 80/1783] Loss: 21.51806 (STFT: 21.5149, VQ: 0.0000) Codes: 13/256 LR: 2.0e-04
[Epoch 1, Batch 160/1783] Loss: 6.69967 (STFT: 6.6949, VQ: 0.0001) Codes: 53/256 LR: 2.0e-04
[Epoch 1, Batch 240/1783] Loss: 11318.51367 (STFT: 11318.5117, VQ: 0.0001) Codes: 27/256 LR: 2.0e-04
[Epoch 1, Batch 320/1783] Loss: 6.38741 (STFT: 6.3826, VQ: 0.0001) Codes: 41/256 LR: 2.0e-04
[Epoch 1, Batch 400/1783] Loss: 7.35436 (STFT: 7.3510, VQ: 0.0001) Codes: 25/256 LR: 2.0e-04
[Epoch 1, Batch 480/1783] Loss: 6401.15723 (STFT: 6401.1533, VQ: 0.0001) Codes: 44/256 LR: 2.0e-04
[Epoch 1, Batch 560/1783] Loss: 5.71190 (STFT: 5.7076, VQ: 0.0001) Codes: 38/256 LR: 2.0e-04
[Epoch 1, Batch 640/1783] Loss: 1544.09180 (STFT: 1544.0887, VQ: 0.0000) Codes: 32/256 LR: 2.0e-04
[Epoch 1, Batch 720/1783] Loss: 6.15839 (STFT: 6.1544, VQ: 0.0001) Codes: 45/256 LR: 2.0e-04
[Epoch 1, Batch 800/1783] Loss: 5.70739 (STFT: 5.7039, VQ: 0.0001) Codes: 51/256 LR: 2.0e-04
[Epoch 1, Batch 880/1783] Loss: 9.99475 (STFT: 9.

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#t

[Epoch 2, Batch 80/1783] Loss: 3.65215 (STFT: 3.6482, VQ: 0.0001) Codes: 52/256 LR: 2.0e-04
[Epoch 2, Batch 160/1783] Loss: 3.89023 (STFT: 3.8857, VQ: 0.0001) Codes: 68/256 LR: 2.0e-04
[Epoch 2, Batch 240/1783] Loss: 3.97062 (STFT: 3.9671, VQ: 0.0001) Codes: 76/256 LR: 2.0e-04
[Epoch 2, Batch 320/1783] Loss: 4.83630 (STFT: 4.8326, VQ: 0.0001) Codes: 61/256 LR: 2.0e-04
[Epoch 2, Batch 400/1783] Loss: 5.67443 (STFT: 5.6715, VQ: 0.0001) Codes: 66/256 LR: 2.0e-04
[Epoch 2, Batch 480/1783] Loss: 7.60566 (STFT: 7.6011, VQ: 0.0001) Codes: 74/256 LR: 2.0e-04
[Epoch 2, Batch 560/1783] Loss: 12.08193 (STFT: 12.0782, VQ: 0.0001) Codes: 37/256 LR: 2.0e-04
[Epoch 2, Batch 640/1783] Loss: 4.89683 (STFT: 4.8928, VQ: 0.0001) Codes: 48/256 LR: 2.0e-04
[Epoch 2, Batch 720/1783] Loss: 4.96130 (STFT: 4.9572, VQ: 0.0001) Codes: 91/256 LR: 2.0e-04
[Epoch 2, Batch 800/1783] Loss: 287.49152 (STFT: 287.4873, VQ: 0.0001) Codes: 48/256 LR: 2.0e-04
[Epoch 2, Batch 880/1783] Loss: 4.24902 (STFT: 4.2453, VQ: 0.0001

...

[Epoch 10, Batch 80/1783] Loss: 543.91187 (STFT: 543.8990, VQ: 0.0049) Codes: 172/256 LR: 2.0e-04
[Epoch 10, Batch 160/1783] Loss: 5544.81982 (STFT: 5544.8120, VQ: 0.0022) Codes: 195/256 LR: 2.0e-04
[Epoch 10, Batch 240/1783] Loss: 3.36313 (STFT: 3.3466, VQ: 0.0062) Codes: 178/256 LR: 2.0e-04
[Epoch 10, Batch 320/1783] Loss: 110.22075 (STFT: 110.2092, VQ: 0.0042) Codes: 159/256 LR: 2.0e-04
[Epoch 10, Batch 400/1783] Loss: 2.62550 (STFT: 2.6165, VQ: 0.0031) Codes: 179/256 LR: 2.0e-04
[Epoch 10, Batch 480/1783] Loss: 3.67330 (STFT: 3.6601, VQ: 0.0049) Codes: 173/256 LR: 2.0e-04
[Epoch 10, Batch 560/1783] Loss: 2.92026 (STFT: 2.9120, VQ: 0.0027) Codes: 172/256 LR: 2.0e-04
[Epoch 10, Batch 640/1783] Loss: 12.53675 (STFT: 12.5182, VQ: 0.0075) Codes: 192/256 LR: 2.0e-04
[Epoch 10, Batch 720/1783] Loss: 4.42895 (STFT: 4.4157, VQ: 0.0056) Codes: 193/256 LR: 2.0e-04
[Epoch 10, Batch 800/1783] Loss: 3.06803 (STFT: 3.0519, VQ: 0.0054) Codes: 190/256 LR: 2.0e-04
[Epoch 10, Batch 880/1783] Loss: 4.

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

[Epoch 11, Batch 80/1783] Loss: 3.39441 (STFT: 3.3731, VQ: 0.0090) Codes: 176/256 LR: 1.4e-04
[Epoch 11, Batch 160/1783] Loss: 21.10280 (STFT: 21.0850, VQ: 0.0071) Codes: 169/256 LR: 1.4e-04
[Epoch 11, Batch 240/1783] Loss: 3.19578 (STFT: 3.1782, VQ: 0.0066) Codes: 176/256 LR: 1.4e-04
[Epoch 11, Batch 320/1783] Loss: 157.62001 (STFT: 157.5966, VQ: 0.0101) Codes: 199/256 LR: 1.4e-04
[Epoch 11, Batch 400/1783] Loss: 2.89894 (STFT: 2.8851, VQ: 0.0049) Codes: 198/256 LR: 1.4e-04
[Epoch 11, Batch 480/1783] Loss: 8.41100 (STFT: 8.3914, VQ: 0.0077) Codes: 197/256 LR: 1.4e-04
[Epoch 11, Batch 560/1783] Loss: 3.37289 (STFT: 3.3621, VQ: 0.0038) Codes: 173/256 LR: 1.4e-04
[Epoch 11, Batch 640/1783] Loss: 3.51621 (STFT: 3.4938, VQ: 0.0091) Codes: 206/256 LR: 1.4e-04
[Epoch 11, Batch 720/1783] Loss: 3.04318 (STFT: 3.0277, VQ: 0.0061) Codes: 200/256 LR: 1.4e-04
[Epoch 11, Batch 800/1783] Loss: 3.43393 (STFT: 3.4123, VQ: 0.0094) Codes: 180/256 LR: 1.4e-04
[Epoch 11, Batch 880/1783] Loss: 3.14675 (STF

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#t

[Epoch 12, Batch 80/1783] Loss: 2.90751 (STFT: 2.8842, VQ: 0.0096) Codes: 198/256 LR: 1.4e-04

!!! KeyboardInterrupt received. Stopping... !!!
--- Epoch 12 finished --- Average Loss: 122.94958 ---
--- LR: 1.4e-04 ---
Training stopped by user.
Training stopped.
Final model saved.
--- Training Finished ---

Run the next cell to download your model.


In [ ]:
# ---------------------------------------------------------------------
# CELL 1: PASTE ALL OF THIS CODE INTO THE FIRST COLAB CELL AND RUN IT
# ---------------------------------------------------------------------

#@title A-to-Z Training Script for Neural Codec (TS3 GAN)
#@markdown ### 1. Install Dependencies
#@markdown This cell will install all required packages and define all models and training functions.
!pip install pesq[speechmetrics] pystoi librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import os
import numpy as np
import torch.nn.functional as F
import math
import traceback
import tarfile
import requests
import threading
import librosa # Added for metric calculation in validation
from pesq import pesq # Added for metric calculation in validation
from pystoi import stoi # Added for metric calculation in validation
from torch.optim.lr_scheduler import ExponentialLR
from google.colab import files # For downloading

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---------------------------------------------------------------------
# CONFIGURATION OPTIONS
# ---------------------------------------------------------------------
#@markdown ---
#@markdown ### 2. Training Configuration
#@markdown Training uses the Adversarial Codec (GACodec) for high quality.
MODEL_ARCHITECTURE = "TS3_Codec (16kbps, Transformer GAN)" #@param ["TS3_Codec (16kbps, Transformer GAN)"]
DATASET_TO_DOWNLOAD = "train-clean-100" #@param ["dev-clean", "train-clean-100", "train-clean-360"]
#@markdown *Note: Use `train-clean-100` for best results. Training GANs requires larger data.*

#@markdown ---
#@markdown ### 3. Training Hyperparameters (CRITICAL TUNING FOR PESQ > 3.6)
EPOCHS = 1000 #@param {type:"integer"}
#@markdown *Set to 1000 for realistic high-quality training. Will be stopped by session limit.*
LEARNING_RATE = 0.0001 #@param {type:"number"}

#@markdown **Adversarial and Feature Loss Weights**
#@markdown *These are key for quality.*
LAMBDA_STFT = 45.0 #@param {type:"number"}
LAMBDA_FM = 4.0 #@param {type:"number"}
#@markdown *(Increased from 2.0 to 4.0 to force better perceptual feature matching)*

#@markdown **GAN Stability Controls**
DISC_WARMUP_STEPS = 100 #@param {type:"integer"}
LR_DECAY_RATE = 0.9999 #@param {type:"number"}
#@markdown **Discriminator to Generator Training Ratio**
D_G_RATIO = 2 #@param {type:"integer"}
#@markdown *(Train Discriminator 2x for every 1x Generator to maintain balance)*

#@markdown ---
#@markdown ### 4. Hardware & VRAM Settings
BATCH_SIZE = 8 #@param {type:"integer"}
#@markdown *Note: Increased model complexity may require even smaller batch sizes (4 or 6).*
GRAD_ACCUM_STEPS = 4 #@param {type:"integer"}
USE_AMP = True #@param {type:"boolean"}
NUM_WORKERS = 4 #@param {type:"integer"}

#@markdown **Latency Control (Crucial for <20ms end-to-end)**
TFM_HISTORY_CHUNKS = 0 #@param {type:"integer"}
#@markdown *SET TO 0: Guarantees 10ms frame latency (160 samples). Set to 1 (20ms total) or 2 (30ms total) if quality plateau is hit.*

#@markdown ---
#@markdown ### 5. File Paths
SAVE_PATH_PREFIX = "low_latency_codec" #@param {type:"string"}
DOWNLOAD_PATH = "/content/LibriSpeech" #@param {type:"string"}
#@markdown ---

# ---------------------------------------------------------------------
# DATASET DOWNLOAD FUNCTION (Modified to download validation set)
# ---------------------------------------------------------------------

def download_librispeech(dataset_name, path):
    """Downloads and extracts a LibriSpeech dataset."""
    url_map = {
        "dev-clean": "https://www.openslr.org/resources/12/dev-clean.tar.gz",
        "train-clean-100": "https://www.openslr.org/resources/12/train-clean-100.tar.gz",
        "train-clean-360": "https://www.openslr.org/resources/12/train-clean-360.tar.gz",
    }

    # We must ensure dev-clean is downloaded for validation, regardless of main choice
    datasets_to_check = [dataset_name]
    if dataset_name != "dev-clean":
        datasets_to_check.append("dev-clean")

    downloaded_paths = {}

    for name in datasets_to_check:
        if name not in url_map: continue

        url = url_map[name]
        filename = url.split("/")[-1]
        compressed_path = os.path.join(path, filename)
        target_folder_name = name
        expected_dataset_path = os.path.join(path, "LibriSpeech", target_folder_name)

        print(f"Checking for existing dataset folder for {name} at: {expected_dataset_path}")
        if os.path.exists(expected_dataset_path) and os.path.isdir(expected_dataset_path):
            print(f"Dataset {name} already found.")
            downloaded_paths[name] = expected_dataset_path
            continue

        try:
            if not os.path.exists(compressed_path):
                print(f"Downloading {filename}...")
                with requests.get(url, stream=True) as r:
                    r.raise_for_status()

                    # FIX: Ensure parent directory for compressed file exists
                    os.makedirs(os.path.dirname(compressed_path), exist_ok=True)

                    with open(compressed_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            f.write(chunk)
                print("Download complete.")

            print(f"Extracting {name}...")
            with tarfile.open(compressed_path, "r:gz") as tar:
                tar.extractall(path=path)
            print("Extraction complete.")
            os.remove(compressed_path)

            if os.path.exists(expected_dataset_path) and os.path.isdir(expected_dataset_path):
                print(f"✓ Dataset {name} ready.")
                downloaded_paths[name] = expected_dataset_path
            else:
                print(f"ERROR: Could not find extracted folder for {name}.")

        except Exception as e:
            print(f"Error during download/extraction for {name}: {e}")
            traceback.print_exc()

    if dataset_name not in downloaded_paths:
        return None, None

    return downloaded_paths.get(dataset_name), downloaded_paths.get("dev-clean")


# ---------------------------------------------------------------------
# MODEL DEFINITIONS (OPTIMIZED FOR CPU SPEED)
# ---------------------------------------------------------------------

# --- Global Configuration (Unchanged) ---
HOP_SIZE = 160 # 10ms frame (160 samples / 16000 Hz = 0.01s) - CRITICAL LATENCY FIX
LATENT_DIM = 128 # High capacity

VQ_EMBEDDINGS = 256
NUM_VQ_STAGES = 2
VQ_INDICES_PER_STAGE = 10
NUM_QUANTIZERS = VQ_INDICES_PER_STAGE

# --- Loss Components (Unchanged) ---
class MultiResolutionSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[1024, 2048, 512], hop_sizes=[120, 240, 50], win_lengths=[600, 1200, 240]):
        super(MultiResolutionSTFTLoss, self).__init__()
        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        self.win_lengths = win_lengths
        self.window = torch.hann_window
        assert len(fft_sizes) == len(hop_sizes) == len(win_lengths)

    def forward(self, y_hat, y):
        sc_loss, mag_loss = 0.0, 0.0
        for fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            window = self.window(win, device=y.device)
            y_hat_float = y_hat.squeeze(1).to(torch.float32)
            y_float = y.squeeze(1).to(torch.float32)
            spec_hat = torch.stft(y_hat_float, n_fft=fft, hop_length=hop, win_length=win, window=window, return_complex=True)
            spec = torch.stft(y_float, n_fft=fft, hop_length=hop, win_length=win, window=window, return_complex=True)

            sc_loss += torch.norm(torch.abs(spec) - torch.abs(spec_hat), p='fro') / (torch.norm(torch.abs(spec), p='fro') + 1e-6)
            mag_loss += F.l1_loss(torch.log(torch.abs(spec).clamp(min=1e-9)), torch.log(torch.abs(spec_hat).clamp(min=1e-9)))

        return (sc_loss / len(self.fft_sizes)) + (mag_loss / len(self.fft_sizes))

# --- Causal Components (Unchanged) ---
class CausalConv1d(nn.Conv1d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.causal_padding = self.kernel_size[0] - 1
    def forward(self, x):
        return super().forward(F.pad(x, (self.causal_padding, 0)))

class CausalConvTranspose1d(nn.ConvTranspose1d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.causal_padding = self.kernel_size[0] - self.stride[0]
    def forward(self, x):
        x = super().forward(x)
        if self.causal_padding != 0:
            return x[..., :-self.causal_padding]
        return x

# --- Residual Vector Quantizer (RVQ) (Unchanged) ---
class ResidualVectorQuantizer(nn.Module):
    def __init__(self, num_stages, num_embeddings, embedding_dim, commitment_cost):
        super().__init__()
        self.num_stages = num_stages
        self.commitment_cost = commitment_cost
        self.vqs = nn.ModuleList([
            VectorQuantizerSingle(num_embeddings, embedding_dim, 0.0)
            for _ in range(num_stages)
        ])

    def forward(self, z_e):
        quantized_output = 0.0
        residual = z_e
        total_vq_loss = 0.0
        all_indices = []

        for vq in self.vqs:
            z_q_i, vq_loss_i, indices_i = vq(residual)
            residual = residual - z_q_i.detach()
            quantized_output = quantized_output + z_q_i
            total_vq_loss += vq_loss_i
            all_indices.append(indices_i)

        total_vq_loss = total_vq_loss * self.commitment_cost / self.num_stages
        return quantized_output, total_vq_loss, torch.stack(all_indices, dim=1)

class VectorQuantizerSingle(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / self.num_embeddings, 1.0 / self.num_embeddings)

    def forward(self, z_e):
        z_e_float = z_e.to(torch.float32)
        z_e_flat = z_e_float.permute(0, 2, 1).contiguous().view(-1, self.embedding_dim)
        distances = (torch.sum(z_e_flat**2, dim=1, keepdim=True)
                     + torch.sum(self.embedding.weight**2, dim=1)
                     - 2 * torch.matmul(z_e_flat, self.embedding.weight.t()))
        encoding_indices = torch.argmin(distances, dim=1)
        z_q = self.embedding(encoding_indices).view(z_e_float.shape[0], -1, self.embedding_dim)
        z_q = z_q.permute(0, 2, 1).contiguous()
        e_loss = F.mse_loss(z_q.detach(), z_e_float)
        z_q = z_e + (z_q - z_e_float).detach()
        return z_q, e_loss, encoding_indices.view(z_e.shape[0], -1)


# --- Encoder/Decoder (Unchanged - Architecturally Sound for 16kbps/10ms) ---
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        ResBlock = lambda c: nn.Sequential(CausalConv1d(c, c, 3), nn.ELU(), CausalConv1d(c, c, 1))
        # Total Downsampling: 16x. (160 samples -> 10 latent vectors)
        self.net = nn.Sequential(
            CausalConv1d(1, 64, 5), nn.ELU(), ResBlock(64),
            CausalConv1d(64, 128, 3, stride=2), nn.ELU(), ResBlock(128), # x2
            CausalConv1d(128, 256, 3, stride=2), nn.ELU(), ResBlock(256), # x4
            CausalConv1d(256, 512, 3, stride=2), nn.ELU(), ResBlock(512), # x8
            CausalConv1d(512, 512, 3, stride=2), nn.ELU(), ResBlock(512), # x16 total stride
            CausalConv1d(512, LATENT_DIM, 3, stride=1), nn.ELU(),
        )
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        ResBlock = lambda c: nn.Sequential(CausalConv1d(c, c, 3), nn.ELU(), CausalConv1d(c, c, 1))
        # Total Upsampling: x16. (10 latent vectors -> 160 samples)
        self.net = nn.Sequential(
            CausalConvTranspose1d(LATENT_DIM, 512, 3, stride=1), nn.ELU(), ResBlock(512),
            CausalConvTranspose1d(512, 512, 3, stride=2), nn.ELU(), ResBlock(512), # 10 -> 20
            CausalConvTranspose1d(512, 256, 3, stride=2), nn.ELU(), ResBlock(256), # 20 -> 40
            CausalConvTranspose1d(256, 128, 3, stride=2), nn.ELU(), ResBlock(128), # 40 -> 80
            CausalConvTranspose1d(128, 64, 3, stride=2), nn.ELU(), ResBlock(64), # 80 -> 160 (x16 total)
            CausalConv1d(64, 1, 3), nn.Tanh()
        )
    def forward(self, x): return self.net(x)

# --- Causal Transformer (CRITICAL CPU LATENCY FIX) ---
class CausalTransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, num_layers=1, history_chunks=0):
        super().__init__()
        self.d_model = d_model
        # CRITICAL FIX: Reduced to 1 layer for max CPU performance
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, activation=F.gelu)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.state_len = history_chunks * VQ_INDICES_PER_STAGE
        print(f"CausalTransformerEncoder initialized with STATE_LEN: {self.state_len}, Layers: {num_layers}")


    def get_causal_mask(self, sz, device):
        return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).to(torch.bool)

    def forward(self, x, state):
        if self.state_len > 0 and state is not None:
            state_history = state[:, -self.state_len:, :]
            inp = torch.cat([state_history, x], dim=1)
        else:
            inp = x

        new_state = inp.detach()
        mask = self.get_causal_mask(inp.shape[1], x.device)
        norm_inp = self.norm(inp)
        out = self.transformer(norm_inp, mask=mask, src_key_padding_mask=None)
        out = out[:, -x.shape[1]:, :]
        return out, new_state


# --- Generator: TS3 Codec (OPTIMIZED RVQ Version) ---
class TS3_Codec(nn.Module):
    """The Generator model now utilizing RVQ for increased capacity."""
    def __init__(self, history_chunks=0):
        super().__init__()
        self.encoder = Encoder()
        self.quantizer = ResidualVectorQuantizer(NUM_VQ_STAGES, VQ_EMBEDDINGS, LATENT_DIM, 0.25)
        # CRITICAL FIX: Reduced TFM layers from 2 to 1 for max CPU speed.
        self.encoder_tfm = CausalTransformerEncoder(LATENT_DIM, nhead=4, num_layers=1, history_chunks=history_chunks)
        self.decoder_tfm = CausalTransformerEncoder(LATENT_DIM, nhead=4, num_layers=1, history_chunks=history_chunks)
        self.decoder = Decoder()

    def forward(self, x, h_enc=None, h_dec=None):
        x = x.to(torch.float32)
        z_e = self.encoder(x)
        z_e_tfm_in = z_e.permute(0, 2, 1)
        z_e_tfm_out, h_enc_new = self.encoder_tfm(z_e_tfm_in, h_enc)
        z_e_tfm_out = z_e_tfm_out.permute(0, 2, 1)
        z_q, vq_loss, indices = self.quantizer(z_e_tfm_out)
        z_q_tfm_in = z_q.permute(0, 2, 1)
        z_q_tfm_out, h_dec_new = self.decoder_tfm(z_q_tfm_in, h_dec)
        z_q_tfm_out = z_q_tfm_out.permute(0, 2, 1)
        x_hat = self.decoder(z_q_tfm_out)
        return x_hat, vq_loss, (h_enc_new, h_dec_new)

    def encode(self, x, h_enc):
        """Streaming encode path."""
        x = x.to(torch.float32)
        z_e = self.encoder(x)
        z_e_tfm_in = z_e.permute(0, 2, 1)
        z_e_tfm_out, h_enc_new = self.encoder_tfm(z_e_tfm_in, h_enc)
        z_e_tfm_out = z_e_tfm_out.permute(0, 2, 1)
        with torch.no_grad():
            _, _, indices = self.quantizer(z_e_tfm_out)
        return indices.view(indices.shape[0], -1), h_enc_new

    def decode(self, indices, h_dec):
        """Streaming decode path - reconstructs VQ vectors from flattened indices."""
        indices = indices.view(indices.shape[0], NUM_VQ_STAGES, VQ_INDICES_PER_STAGE)
        z_q = 0.0
        for stage in range(NUM_VQ_STAGES):
            vq_layer = self.quantizer.vqs[stage]
            indices_i = indices[:, stage, :]
            z_q_i = vq_layer.embedding(indices_i)
            z_q = z_q + z_q_i.permute(0, 2, 1)

        z_q_tfm_in = z_q.permute(0, 2, 1)
        z_q_tfm_out, h_dec_new = self.decoder_tfm(z_q_tfm_in, h_dec)
        z_q_tfm_out = z_q_tfm_out.permute(0, 2, 1)
        x_hat = self.decoder(z_q_tfm_out)
        return x_hat, h_dec_new


# --- Discriminator: Multi-Scale Adversarial Network (Unchanged) ---
class Discriminator(nn.Module):
    """Single Discriminator operating on one scale/resolution."""
    def __init__(self, start_channels=16):
        super().__init__()
        self.net = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(1, start_channels, 15, stride=1, padding=7),
                nn.LeakyReLU(0.2),
            ),
            nn.Sequential(nn.Conv1d(start_channels, start_channels * 2, 41, stride=4, padding=20, groups=4), nn.LeakyReLU(0.2)),
            nn.Sequential(nn.Conv1d(start_channels * 2, start_channels * 4, 41, stride=4, padding=20, groups=16), nn.LeakyReLU(0.2)),
            nn.Sequential(nn.Conv1d(start_channels * 4, start_channels * 8, 41, stride=4, padding=20, groups=64), nn.LeakyReLU(0.2)),
            nn.Sequential(nn.Conv1d(start_channels * 8, start_channels * 8, 5, stride=1, padding=2), nn.LeakyReLU(0.2)),
        ])
        self.final_conv = nn.Conv1d(start_channels * 8, 1, 3, stride=1, padding=1)

    def forward(self, x):
        feature_maps = []
        for layer in self.net:
            x = layer(x)
            feature_maps.append(x)
        output = self.final_conv(x)
        feature_maps.append(output)
        return output, feature_maps

class MultiScaleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.discriminators = nn.ModuleList([
            Discriminator(start_channels=16),
            Discriminator(start_channels=32),
            Discriminator(start_channels=64),
        ])
        self.downsample = nn.AvgPool1d(kernel_size=4, stride=2, padding=1, count_include_pad=False)

    def forward(self, x):
        outputs = []
        feature_maps = []

        out_d1, fm_d1 = self.discriminators[0](x)
        outputs.append(out_d1); feature_maps.append(fm_d1)

        x_2x = self.downsample(x)
        out_d2, fm_d2 = self.discriminators[1](x_2x)
        outputs.append(out_d2); feature_maps.append(fm_d2)

        x_4x = self.downsample(x_2x)
        out_d3, fm_d3 = self.discriminators[2](x_4x)
        outputs.append(out_d3); feature_maps.append(fm_d3)

        return outputs, feature_maps


# --- TRADITIONAL CODECS (Unchanged) ---
class MuLawCodec:
    def __init__(self, quantization_channels=256): self.mu = float(quantization_channels - 1)
    def encode(self, x):
        mu_t = torch.tensor(self.mu, device=x.device, dtype=torch.float32)
        encoded = torch.sign(x) * torch.log1p(mu_t * torch.abs(x)) / torch.log1p(mu_t)
        return (((encoded + 1) / 2 * self.mu) + 0.5).to(torch.uint8)
    def decode(self, z):
        z_float = z.to(torch.float32)
        mu_t = torch.tensor(self.mu, device=z.device, dtype=torch.float32)
        y = (z_float / self.mu) * 2.0 - 1.0
        return (torch.sign(y) * (1.0 / self.mu) * (torch.pow(1.0 + self.mu, torch.abs(y)) - 1.0)).unsqueeze(1)

class ALawCodec:
    def __init__(self): self.A = 87.6
    def encode(self, x):
        a_t = torch.tensor(self.A, device=x.device, dtype=torch.float32)
        abs_x = torch.abs(x)
        encoded = torch.zeros_like(x)
        cond = abs_x < (1 / self.A)
        encoded[cond] = torch.sign(x[cond]) * (a_t * abs_x[cond]) / (1 + torch.log(a_t))
        encoded[~cond] = torch.sign(x[~cond]) * (1 + torch.log(a_t * abs_x[~cond])) / (1 + torch.log(a_t))
        return (((encoded + 1) / 2 * 255) + 0.5).to(torch.uint8)
    def decode(self, z):
        z_float = z.to(torch.float32)
        a_t = torch.tensor(self.A, device=z.device, dtype=torch.float32)
        y = (z_float / 127.5) - 1.0
        abs_y = torch.abs(y)
        decoded = torch.zeros_like(y)
        cond = abs_y < (1 / (1 + torch.log(a_t)))
        decoded[cond] = torch.sign(y[cond]) * (abs_y[cond] * (1 + torch.log(a_t))) / a_t
        decoded[~cond] = torch.sign(y[~cond]) * torch.exp(abs_y[~cond] * (1 + torch.log(a_t)) - 1) / a_t
        return decoded.unsqueeze(1)

# --- DATASET & TRAINING LOGIC (Modified for Validation Data) ---
TRAIN_CHUNK_SIZE = 16000

class AudioChunkDataset(Dataset):
    def __init__(self, directory, chunk_size=TRAIN_CHUNK_SIZE, sample_rate=16000):
        self.chunk_size, self.sample_rate = chunk_size, sample_rate
        self.file_paths = [os.path.join(r, f) for r, _, fs in os.walk(directory) for f in fs if f.lower().endswith(('.wav', '.flac'))]
        if not self.file_paths: raise ValueError("No audio files found.")
    def __len__(self): return len(self.file_paths)
    def __getitem__(self, idx):
        try:
            waveform, sr = torchaudio.load(self.file_paths[idx])
            if sr != self.sample_rate: waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform)
            if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            waveform = waveform.to(torch.float32)
            if waveform.shape[1] > self.chunk_size:
                start = np.random.randint(0, waveform.shape[1] - self.chunk_size)
                waveform = waveform[:, start:start + self.chunk_size]
            else:
                waveform = F.pad(waveform, (0, self.chunk_size - waveform.shape[1]))
            if waveform.shape[1] != self.chunk_size:
                waveform = F.pad(waveform, (0, self.chunk_size - waveform.shape[1]))
            return waveform
        except Exception as e:
            print(f"Warning: Skipping file {self.file_paths[idx]}. Error: {e}")
            return torch.zeros((1, self.chunk_size), dtype=torch.float32)


def generator_loss(disc_fake_features, disc_real_features_for_fm, fm_weight, gan_weight):
    """Calculates Generator Loss (Adversarial + Feature Matching)."""
    adv_loss = 0.0
    fm_loss = 0.0

    # Adversarial Loss (LSGAN/MSE Loss for Generator: Target real=1.0)
    for disc_fake in disc_fake_features:
        adv_loss += F.mse_loss(disc_fake[-1], torch.ones_like(disc_fake[-1]))

    # Feature Matching Loss (L1 distance between intermediate discriminator features)
    for (fake_features, real_features) in zip(disc_fake_features, disc_real_features_for_fm):
        for (fake_feature, real_feature) in zip(fake_features[:-1], real_features[:-1]):
            fm_loss += F.l1_loss(fake_feature, real_feature.detach())

    # Normalize losses
    num_disc = len(disc_fake_features)
    num_fm_layers = sum(len(f) - 1 for f in disc_fake_features)
    fm_loss = fm_loss / (num_fm_layers if num_fm_layers > 0 else 1.0)
    adv_loss = adv_loss / num_disc

    return adv_loss * gan_weight, fm_loss * fm_weight

def discriminator_loss(disc_real_features, disc_fake_features):
    """Calculates Discriminator Loss (Real vs Fake)."""
    loss = 0.0
    # LSGAN/MSE Loss for Discriminator: Target real=1.0, target fake=0.0
    for real_out, fake_out in zip(disc_real_features, disc_fake_features):
        real_loss = F.mse_loss(real_out[-1], torch.ones_like(real_out[-1]))
        fake_loss = F.mse_loss(fake_out[-1], torch.zeros_like(fake_out[-1]))
        loss += (real_loss + fake_loss)
    return loss / len(disc_real_features)

# --- NEW: Metric Calculation Utility (Uses Val DataLoader) ---
def calculate_metrics(model, dataloader, device, progress_callback, sr=16000):
    """Calculates PESQ and STOI on a subset of the DEV data."""
    model.eval()
    total_pesq, total_stoi, total_samples = 0, 0, 0
    max_test_batches = 10 # Limit test batches for speed

    test_data_iter = iter(dataloader)

    try:
        for batch_idx in range(max_test_batches):
            inputs = next(test_data_iter).to(device)

            with torch.no_grad():
                h_enc, h_dec = None, None
                x_hat, _, _ = model(inputs, h_enc, h_dec)

                x_hat = x_hat[..., :inputs.shape[-1]]

                original_np = inputs.squeeze().cpu().numpy()
                reconstructed_np = x_hat.squeeze().cpu().numpy()

                for i in range(original_np.shape[0]):
                    orig = original_np[i]
                    rec = reconstructed_np[i]

                    try:
                        pesq_score = pesq(sr, orig, rec, 'wb')
                        stoi_score = stoi(orig, rec, sr, extended=False)

                        total_pesq += pesq_score
                        total_stoi += stoi_score
                        total_samples += 1
                    except Exception as e:
                        progress_callback.emit(f"Warning: Metric calculation failed for one sample: {e}")
                        continue

        if total_samples > 0:
            avg_pesq = total_pesq / total_samples
            avg_stoi = total_stoi / total_samples
            progress_callback.emit(f"--- VALIDATION (DEV SET) --- PESQ: {avg_pesq:.3f} | STOI: {avg_stoi:.3f} ---")

        model.train()
    except StopIteration:
        progress_callback.emit("Warning: Reached end of validation dataloader during metric calculation.")
        model.train()


def train_model(train_dataset_path, val_dataset_path, epochs, learning_rate, batch_size, model_save_path, progress_callback, stop_event, model_type, use_amp=True, accum_steps=1, num_workers=4, disc_warmup_steps=100, lr_decay_rate=0.9999, tfm_history_chunks=0, lambda_stft=45.0, lambda_fm=4.0, d_g_ratio=2):
    """
    Main GAN Training function with stability controls and validation.
    """
    if model_type != 'transformer':
        progress_callback.emit("Internal Error: Training only supports 'transformer' model_type now."); return

    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        progress_callback.emit(f"Using device: {device}")

        # Training Data Setup
        train_dataset = AudioChunkDataset(directory=train_dataset_path)
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
        progress_callback.emit(f"Train Dataset loaded with {len(train_dataset)} files.")

        # Validation Data Setup (Critical Fix)
        val_dataset = AudioChunkDataset(directory=val_dataset_path)
        # Use a large batch size for faster validation metrics
        val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=1, pin_memory=True, drop_last=True)
        progress_callback.emit(f"Validation Dataset (dev-clean) loaded with {len(val_dataset)} files.")


        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        generator = TS3_Codec(history_chunks=tfm_history_chunks).to(device)
        discriminator = MultiScaleDiscriminator().to(device)

        optimizer_g = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.9))
        optimizer_d = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.9))

        scheduler_g = ExponentialLR(optimizer_g, gamma=lr_decay_rate)
        scheduler_d = ExponentialLR(optimizer_d, gamma=lr_decay_rate)

        stft_criterion = MultiResolutionSTFTLoss().to(device)
        l1_criterion = nn.L1Loss().to(device)

        progress_callback.emit(f"Starting GAN training for TS3_Codec (Generator) and Discriminator...")

        lambda_l1, lambda_vq = 1.0, 0.1
        gan_weight, fm_weight = 1.0, lambda_fm
        global_step = 0
        d_step_counter = 0

        # Initial validation check (Epoch 0)
        calculate_metrics(generator, val_dataloader, device, progress_callback)

        for epoch in range(epochs):
            if stop_event.is_set():
                progress_callback.emit("Training stopped by user."); break

            generator.train()
            discriminator.train()

            for i, data in enumerate(train_dataloader):
                if stop_event.is_set(): break
                inputs = data.to(device)
                global_step += 1
                d_step_counter += 1

                # --- 2. TRAIN DISCRIMINATOR (MSD) ---
                if d_step_counter % d_g_ratio == 0:
                    optimizer_d.zero_grad()

                    with torch.cuda.amp.autocast(enabled=use_amp):
                        h_enc, h_dec = None, None
                        x_hat, _, _ = generator(inputs, h_enc, h_dec)
                        input_len = inputs.shape[-1]
                        if x_hat.shape[-1] > input_len:
                            x_hat = x_hat[..., :input_len]
                        elif x_hat.shape[-1] < input_len:
                            x_hat = F.pad(x_hat, (0, input_len - x_hat.shape[-1]))

                        x_hat_detached = x_hat.detach()

                        _, disc_real_features = discriminator(inputs)
                        _, disc_fake_features = discriminator(x_hat_detached)
                        loss_d_total = discriminator_loss(disc_real_features, disc_fake_features)

                    scaler.scale(loss_d_total / accum_steps).backward()

                    if global_step % accum_steps == 0:
                        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
                        scaler.step(optimizer_d)
                        scaler.update()
                        optimizer_d.zero_grad()
                        scheduler_d.step()

                # --- 1. TRAIN GENERATOR (TS3_Codec) ---
                if global_step > disc_warmup_steps:
                    optimizer_g.zero_grad()

                    with torch.cuda.amp.autocast(enabled=use_amp):
                        x_hat, vq_loss, _ = generator(inputs, h_enc, h_dec)

                        input_len = inputs.shape[-1]
                        if x_hat.shape[-1] > input_len:
                            x_hat = x_hat[..., :input_len]
                        elif x_hat.shape[-1] < input_len:
                            x_hat = F.pad(x_hat, (0, input_len - x_hat.shape[-1]))

                        stft_loss = stft_criterion(x_hat, inputs)
                        l1_loss = l1_criterion(x_hat, inputs)

                        disc_fake_outputs_g, disc_fake_features_g = discriminator(x_hat)
                        with torch.no_grad():
                            _, disc_real_features_for_fm = discriminator(inputs)

                        adv_loss, fm_loss = generator_loss(disc_fake_features_g, disc_real_features_for_fm, fm_weight=fm_weight, gan_weight=gan_weight)

                        loss_g_total = ((adv_loss + fm_loss) +
                                        lambda_stft * stft_loss +
                                        lambda_l1 * l1_loss +
                                        lambda_vq * vq_loss)

                    scaler.scale(loss_g_total / accum_steps).backward()

                    if global_step % accum_steps == 0:
                        torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
                        scaler.step(optimizer_g)
                        scaler.update()
                        optimizer_g.zero_grad()
                        scheduler_g.step()

                    if global_step % (20 * accum_steps * d_g_ratio) == 19:
                        progress_callback.emit(
                            f"[E {epoch + 1}, B {i + 1}] G Loss: {loss_g_total.item():.4f} (Adv: {adv_loss.item():.4f}, FM: {fm_loss.item():.4f}, STFT: {stft_loss.item():.4f}) | D Loss: {loss_d_total.item():.4f} | LR: {optimizer_g.param_groups[0]['lr']:.2e}"
                        )

            # --- CRITICAL VALIDATION STEP ---
            progress_callback.emit(f"--- Epoch {epoch + 1} finished ---")
            if (epoch + 1) % 10 == 0:
                 calculate_metrics(generator, val_dataloader, device, progress_callback)

            # --- CRITICAL CHECKPOINT SAVE ---
            progress_callback.emit(f"Saving epoch {epoch + 1} checkpoint to {model_save_path}...")
            torch.save(generator.state_dict(), model_save_path)
            progress_callback.emit(f"Checkpoint saved.")


        if not stop_event.is_set():
            progress_callback.emit("Training finished. Saving Generator (TS3_Codec)...")
            torch.save(generator.state_dict(), model_save_path)
            progress_callback.emit(f"Generator saved to {model_save_path}")
    except Exception as e:
        progress_callback.emit(f"ERROR in training: {e}")
        progress_callback.emit(traceback.format_exc())

    finally:
        if 'generator' in locals() and generator is not None:
             try:
                 progress_callback.emit(f"Attempting final save of Generator to {model_save_path}...")
                 torch.save(generator.state_dict(), model_save_path)
                 progress_callback.emit(f"Final Generator state saved successfully.")
             except Exception as final_save_e:
                 progress_callback.emit(f"!!! Error during final save: {final_save_e} !!!")


# ---------------------------------------------------------------------
# MAIN EXECUTION SCRIPT
# ---------------------------------------------------------------------
#@markdown ---
#@markdown ### 6. Run Training
#@markdown Click the "Run" button below (or Shift+Enter in this cell) to start the training process using the settings above.
print("Starting main training script...")

# 1. Map GUI string to internal model_type
model_type_internal = "transformer"

# 2. Update save path
final_save_path = f"{SAVE_PATH_PREFIX}_ts3_gacodec.pth"
print(f"Selected model: {MODEL_ARCHITECTURE}")
print(f"Model (Generator) will be saved to: {final_save_path}")

# 3. Download data (fetches main data and dev-clean)
print(f"Downloading/Verifying data in {DOWNLOAD_PATH}...")
train_dataset_path, val_dataset_path = download_librispeech(DATASET_TO_DOWNLOAD, DOWNLOAD_PATH)

if train_dataset_path and val_dataset_path:
    print(f"Train Dataset ready at: {train_dataset_path}")
    print(f"Validation Dataset ready at: {val_dataset_path}")

    # 4. Start training
    print("--- Starting Training ---")
    stop_event = threading.Event()

    class ColabProgressEmitter:
        def emit(self, msg):
            print(msg)

    colab_printer = ColabProgressEmitter()

    train_model(
        train_dataset_path=train_dataset_path,
        val_dataset_path=val_dataset_path, # Pass the separate validation path
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        model_save_path=final_save_path,
        progress_callback=colab_printer,
        stop_event=stop_event,
        model_type=model_type_internal,
        use_amp=USE_AMP,
        accum_steps=GRAD_ACCUM_STEPS,
        num_workers=NUM_WORKERS,
        # Pass new stability params
        disc_warmup_steps=DISC_WARMUP_STEPS,
        lr_decay_rate=LR_DECAY_RATE,
        tfm_history_chunks=TFM_HISTORY_CHUNKS, # Pass latency control
        # Pass new quality params
        lambda_stft=LAMBDA_STFT,
        lambda_fm=LAMBDA_FM,
        d_g_ratio=D_G_RATIO
    )

    print("--- Training Finished ---")
    print(f"Model saved to {final_save_path}")
    print("\n=======================================================")
    print("To download your high-quality Generator model, run the next cell.")
    print("=======================================================")

else:
    print("Failed to download/find dataset. Training aborted.")


PyTorch Version: 2.8.0+cu126
CUDA Available: True
GPU: Tesla T4
Starting main training script...
Selected model: TS3_Codec (16kbps, Transformer GAN)
Model (Generator) will be saved to: low_latency_codec_ts3_gacodec.pth
Downloading/Verifying data in /content/LibriSpeech...
Checking for existing dataset folder for train-clean-100 at: /content/LibriSpeech/LibriSpeech/train-clean-100
Dataset train-clean-100 already found.
Checking for existing dataset folder for dev-clean at: /content/LibriSpeech/LibriSpeech/dev-clean
Dataset dev-clean already found.
Train Dataset ready at: /content/LibriSpeech/LibriSpeech/train-clean-100
Validation Dataset ready at: /content/LibriSpeech/LibriSpeech/dev-clean
--- Starting Training ---
Using device: cuda
Train Dataset loaded with 28539 files.
Validation Dataset (dev-clean) loaded with 2703 files.
CausalTransformerEncoder initialized with STATE_LEN: 0, Layers: 1
CausalTransformerEncoder initialized with STATE_LEN: 0, Layers: 1
Starting GAN training for TS3_C

/tmp/ipython-input-1124920995.py:590: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maint

--- VALIDATION (DEV SET) --- PESQ: 1.023 | STOI: 0.456 ---


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#t

[E 1, B 179] G Loss: 159.2690 (Adv: 0.2553, FM: 0.0248, STFT: 3.5312) | D Loss: 0.5010 | LR: 9.98e-05
[E 1, B 339] G Loss: 119.5051 (Adv: 0.2883, FM: 0.0192, STFT: 2.6475) | D Loss: 0.4597 | LR: 9.94e-05
[E 1, B 499] G Loss: 137.5050 (Adv: 0.2836, FM: 0.0296, STFT: 3.0472) | D Loss: 0.4164 | LR: 9.90e-05
[E 1, B 659] G Loss: 115.3878 (Adv: 0.2319, FM: 0.0298, STFT: 2.5569) | D Loss: 0.4277 | LR: 9.86e-05
[E 1, B 819] G Loss: 119.0976 (Adv: 0.3933, FM: 0.0507, STFT: 2.6350) | D Loss: 0.3670 | LR: 9.82e-05
[E 1, B 979] G Loss: 125.6280 (Adv: 0.3638, FM: 0.0364, STFT: 2.7814) | D Loss: 0.4040 | LR: 9.78e-05
[E 1, B 1139] G Loss: 120.6409 (Adv: 0.5376, FM: 0.0307, STFT: 2.6669) | D Loss: 0.5022 | LR: 9.74e-05
[E 1, B 1299] G Loss: 116.4605 (Adv: 0.4624, FM: 0.0397, STFT: 2.5754) | D Loss: 0.3547 | LR: 9.71e-05
[E 1, B 1459] G Loss: 125.6107 (Adv: 0.3777, FM: 0.0360, STFT: 2.7808) | D Loss: 0.4382 | LR: 9.67e-05
[E 1, B 1619] G Loss: 110.4602 (Adv: 0.3763, FM: 0.0423, STFT: 2.4438) | D Loss

...

[E 10, B 76] G Loss: 64.4350 (Adv: 0.4143, FM: 0.0362, STFT: 1.4201) | D Loss: 0.4104 | LR: 4.48e-05
[E 10, B 236] G Loss: 62.9764 (Adv: 0.6713, FM: 0.0395, STFT: 1.3819) | D Loss: 0.3077 | LR: 4.47e-05
[E 10, B 396] G Loss: 63.5487 (Adv: 0.3885, FM: 0.0412, STFT: 1.4009) | D Loss: 0.3067 | LR: 4.45e-05
[E 10, B 556] G Loss: 64.7187 (Adv: 0.5424, FM: 0.0373, STFT: 1.4234) | D Loss: 0.3459 | LR: 4.43e-05
[E 10, B 716] G Loss: 61.7920 (Adv: 0.4337, FM: 0.0305, STFT: 1.3612) | D Loss: 0.3377 | LR: 4.41e-05
[E 10, B 876] G Loss: 67.2164 (Adv: 0.5663, FM: 0.0421, STFT: 1.4782) | D Loss: 0.3030 | LR: 4.40e-05
[E 10, B 1036] G Loss: 71.1171 (Adv: 0.5586, FM: 0.0433, STFT: 1.5650) | D Loss: 0.3105 | LR: 4.38e-05
[E 10, B 1196] G Loss: 68.1659 (Adv: 0.6260, FM: 0.0277, STFT: 1.4988) | D Loss: 0.2905 | LR: 4.36e-05
[E 10, B 1356] G Loss: 61.9364 (Adv: 0.4689, FM: 0.0428, STFT: 1.3631) | D Loss: 0.2961 | LR: 4.34e-05
[E 10, B 1516] G Loss: 61.1708 (Adv: 0.4461, FM: 0.0386, STFT: 1.3467) | D Loss:

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

--- VALIDATION (DEV SET) --- PESQ: 1.416 | STOI: 0.774 ---
Saving epoch 10 checkpoint to low_latency_codec_ts3_gacodec.pth...
Checkpoint saved.


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#t

[E 11, B 29] G Loss: 68.3081 (Adv: 0.5392, FM: 0.0406, STFT: 1.5032) | D Loss: 0.2957 | LR: 4.11e-05
[E 11, B 189] G Loss: 65.6253 (Adv: 0.4793, FM: 0.0384, STFT: 1.4450) | D Loss: 0.2825 | LR: 4.09e-05
[E 11, B 349] G Loss: 63.1092 (Adv: 0.4240, FM: 0.0385, STFT: 1.3902) | D Loss: 0.3636 | LR: 4.07e-05
[E 11, B 509] G Loss: 64.4660 (Adv: 0.4145, FM: 0.0346, STFT: 1.4209) | D Loss: 0.3206 | LR: 4.06e-05
[E 11, B 669] G Loss: 64.8320 (Adv: 0.4319, FM: 0.0331, STFT: 1.4286) | D Loss: 0.2868 | LR: 4.04e-05
[E 11, B 829] G Loss: 63.3228 (Adv: 0.6835, FM: 0.1104, STFT: 1.3868) | D Loss: 0.2976 | LR: 4.03e-05
[E 11, B 989] G Loss: 71.5338 (Adv: 0.5500, FM: 0.0307, STFT: 1.5751) | D Loss: 0.3486 | LR: 4.01e-05
[E 11, B 1149] G Loss: 62.1915 (Adv: 0.5080, FM: 0.0401, STFT: 1.3679) | D Loss: 0.3217 | LR: 3.99e-05
[E 11, B 1309] G Loss: 64.8984 (Adv: 0.6196, FM: 0.0299, STFT: 1.4262) | D Loss: 0.3276 | LR: 3.98e-05
[E 11, B 1469] G Loss: 61.9797 (Adv: 0.4942, FM: 0.0618, STFT: 1.3627) | D Loss: 

In [ ]:
# ---------------------------------------------------------------------
# CELL 3: RUN THIS CELL *AFTER* TRAINING IS FINISHED (OR STOPPED)
#         TO DOWNLOAD YOUR .pth MODEL FILE
# ---------------------------------------------------------------------

#@title Download Your Trained Model

import os
from google.colab import files

# --- Determine the model file to download based on Cell 2's settings ---
# (Duplicating logic slightly to ensure correct filename)
model_type_map_dl = {
    "GRU_Codec (16kbps, Fast)": "gru",
    "TS3_Codec (16kbps, Transformer)": "transformer",
    "ScoreDec Post-Filter (on GRU Codec)": "scoredec"
}
# Use the values from Cell 2's form (these need to be defined in Cell 2 scope)
# If running this cell independently, manually set these variables:
# MODEL_ARCHITECTURE = "TS3_Codec (16kbps, Transformer)" # Or GRU_Codec...
# SAVE_PATH_PREFIX = "low_latency_codec"

try:
    model_type_internal_dl = model_type_map_dl[MODEL_ARCHITECTURE]
    model_filename = f"{SAVE_PATH_PREFIX}_{model_type_internal_dl}.pth"

    print(f"Attempting to download: {model_filename}")

    if os.path.exists(model_filename):
        print("Model file found. Starting download...")
        files.download(model_filename)
        print("Download initiated.")
    else:
        print(f"ERROR: Model file '{model_filename}' not found!")
        print("Did the training in Cell 2 complete successfully or save a checkpoint?")
        print("Check the file browser on the left to see available .pth files.")

except NameError:
    print("ERROR: Could not determine the model filename.")
    print("Please ensure you have run Cell 2 successfully before running this cell,")
    print("OR manually define MODEL_ARCHITECTURE and SAVE_PATH_PREFIX in this cell.")
except Exception as e:
    print(f"An error occurred during download preparation: {e}")

